In [ ]:
import os
cwd=os.getcwd()
os.chdir('/content/drive/My Drive/Colab Notebooks/JantaHackJul25')

In [ ]:
import pandas as pd
train = pd.read_csv('train_yaOffsB.csv')
test = pd.read_csv('test_pFkWwen.csv')

# Data Pre-Processing


## AutoEDA


In [ ]:
#Automated EDA using sweetviz
!pip install sweetviz

In [ ]:
import sweetviz as sv
training_report = sv.analyze(train)

## Null Value Treatment

In [ ]:
#Manual EDA and Null Value Treatment
train.head()

In [ ]:
train.columns

In [ ]:
train.info()
'''Indicates missing values'''

In [ ]:
train.isnull().sum()
#this represents 10% of the total counts

In [ ]:
# deleting all the null valued rows
#train = train.dropna()

In [ ]:
# imputing values; will check if the model doesnt work properly
train.fillna(train.mean(), inplace=True)

## Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train = train.drop(columns=['ID'])

In [ ]:
X = train.drop(columns = ['Crop_Damage'])
y = train['Crop_Damage']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, random_state=100)

## Data Scaling

In [ ]:
# Robust Scalar
from sklearn.preprocessing import RobustScaler

In [ ]:
x_columns = X_train.columns

In [ ]:
list(x_columns)

In [ ]:
scaler = RobustScaler()

X_train[list(x_columns)] = scaler.fit_transform(X_train[list(x_columns)])

X_train.head()

In [ ]:
X_train.shape

## Correlation

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(train.corr(),cmap='Blues',annot=False)

## One Hot Encoding


In [ ]:
#X_train = pd.get_dummies(data=X_train, columns=['Crop_Type', 'Soil_Type', 'Pesticide_Use_Category','Season'])
#y_train = pd.get_dummies(data=y_train, columns=['Crop_Damage'])
#X_test = pd.get_dummies(data=X_test, columns=['Crop_Type', 'Soil_Type', 'Pesticide_Use_Category','Season'])
#y_test = pd.get_dummies(data=y_test, columns=['Crop_Damage'])

# Model Prototyping

## Inital Model Training

In [ ]:
#Random Forest Approach
from sklearn.metrics import accuracy_score

In [ ]:
# Import the model we are using
from sklearn.ensemble import RandomForestClassifier
# Instantiate model 
rf = RandomForestClassifier(n_estimators= 10, random_state=42)
# Train the model on training data
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test,y_pred)
print('Accuracy:', round(accuracy, 2), '%.')
# THIS IS THE BASELINE MODEL

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state = 42)

from pprint import pprint

# Look at parameters used by our current forest
print('Parameters currently in use:\n')
pprint(rf.get_params())

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 200, stop = 1000, num = 10)]
# Number of features to consider at every split
max_features = ['auto', 'sqrt']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 80, num = 11)]
max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
# Method of selecting samples for training each tree
bootstrap = [True, False]

# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}

pprint(random_grid)

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
rf = RandomForestClassifier(random_state = 42)
# Random search of parameters, using 3 fold cross validation, 
# search across 100 different combinations, and use all available cores
rf_random = RandomizedSearchCV(estimator=rf, param_distributions=random_grid,
                              n_iter = 100, scoring='neg_mean_absolute_error', 
                              cv = 3, verbose=2, random_state=42, n_jobs=-1,
                              return_train_score=True)

# Fit the random search model
rf_random.fit(X_train, y_train);

In [ ]:
rf_random.best_params_

In [ ]:
def evaluate(model, test_features, test_labels):
    predictions = model.predict(test_features)
    accuracy = accuracy_score(test_labels,predictions)
    print('Model Performance')
    print('Accuracy = {:0.2f}%.'.format(accuracy))
    return accuracy

base_model = RandomForestClassifier(n_estimators = 10, random_state = 42)
base_model.fit(X_train, y_train)
base_accuracy = evaluate(base_model, X_test, y_test)

In [ ]:
best_random = rf_random.best_estimator_
random_accuracy = evaluate(best_random, X_test, y_test)

In [ ]:
print('Improvement of {:0.2f}%.'.format( 100 * (random_accuracy - base_accuracy) / base_accuracy))

In [ ]:
import joblib
import pickle
filename = 'best_random_FOREST_BIGGER_LEAF.sav'
joblib.dump(best_random, filename)

## Second Model Training

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

In [ ]:
import joblib
import pickle
filename = 'XGBOOST_IMPUTED.sav'
joblib.dump(best_xgboost_random, filename)

## Predictions

In [ ]:
import joblib
best_random_XGBOOST = joblib.load('XGBOOST_IMPUTED.sav')

In [ ]:
ID = test['ID']
test= test.drop(columns=['ID'])

In [ ]:
test.fillna(test.mean(), inplace=True)

In [ ]:
test

In [ ]:
# Robust Scalar
from sklearn.preprocessing import RobustScaler
x_columns_test = test.columns
scaler = RobustScaler()
test[list(x_columns_test)] = scaler.fit_transform(test[list(x_columns_test)])
test.head()

In [ ]:
'''SAVE PREDICTIONS'''
prediction = best_random_XGBOOST.predict(test)

In [ ]:
prediction

In [ ]:
import numpy as np
np.unique(prediction)

In [ ]:
ID = pd.DataFrame(ID, columns=['ID'])

In [ ]:
ID

In [ ]:
ID['Crop_Damage']= list(prediction)

In [ ]:
ID.to_csv('best_random_xgb.csv',index=False)